# Chapter 10: The Seifert-Van Kampen Theorem

**Source Span:** John M. Lee, *Introduction to Topological Manifolds*, Second Edition, Chapter 10, printed pp. 251-276, PDF pp. 269-294.

## Chapter Goal

This chapter turns the fundamental group from an invariant we compute one space at a time into a gluing machine. The Seifert-Van Kampen theorem says that when a path-connected space is assembled from two open, path-connected pieces with path-connected overlap, the fundamental group of the union is assembled from the two piece groups, with exactly the extra identifications forced by loops that already live in the overlap. In computational terms, the theorem is a ledger: take a free product, record which overlap loops entered through the two inclusions, and quotient by the normal closure of those differences.

The notebook below treats that ledger as a visible object. We will draw the decomposition and the induced group diagram, track the two common special cases, use a spanning tree to see why finite graph groups are free, model two-cell attachment as the operation that kills a boundary word, and read compact surface groups directly from polygon words. The goal is not to reproduce the proof line by line. The goal is to make every proof move inspectable: chopping loops into pieces, thickening wedge summands so the theorem applies, collapsing a graph to a tree plus chords, and converting a two-dimensional cell into a relation in a group presentation.

By the end, a finite CW complex should feel like a computable recipe for a group presentation. Its one-skeleton contributes free generators, its two-cells contribute relators, and cells of dimension at least three leave the fundamental group unchanged. For compact surfaces, this recipe explains why the standard polygon words become the familiar presentations for orientable and nonorientable surfaces, and why abelianization is enough to separate the compact connected surfaces in the classification list.


In [ ]:
# geometry-setup:v1
# Machine-managed by scripts/update_notebook_setup.py. Do not edit this cell by hand.

from __future__ import annotations

import json as _geometry_json
import os as _geometry_os
from pathlib import Path as _GeometryPath
import sys as _geometry_sys

GEOMETRY_SETUP = _geometry_json.loads(
    r"""
{
  "colab_url": "https://colab.research.google.com/github/Rah-Rah-Mitra/Geometry/blob/main/Introduction-to-Topological-Manifolds/chapter-10-the-seifert-van-kampen-theorem/10-the-seifert-van-kampen-theorem.ipynb",
  "course_dir": "Introduction-to-Topological-Manifolds",
  "course_title": "Introduction to Topological Manifolds",
  "github_url": "https://github.com/Rah-Rah-Mitra/Geometry/blob/main/Introduction-to-Topological-Manifolds/chapter-10-the-seifert-van-kampen-theorem/10-the-seifert-van-kampen-theorem.ipynb",
  "jupyterlite": false,
  "marker": "geometry-setup:v1",
  "notebook_kind": "lesson",
  "notebook_path": "Introduction-to-Topological-Manifolds/chapter-10-the-seifert-van-kampen-theorem/10-the-seifert-van-kampen-theorem.ipynb",
  "notebook_title": "Chapter 10: The Seifert-Van Kampen Theorem",
  "repository": {
    "branch": "main",
    "name": "Geometry",
    "owner": "Rah-Rah-Mitra",
    "source_url": "https://github.com/Rah-Rah-Mitra/Geometry"
  },
  "requirements": "requirements/topology.txt",
  "runtime_profile": "topology"
}
"""
)


def _geometry_is_colab():
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except Exception:
        return False


def _geometry_is_jupyterlite():
    return _geometry_sys.platform == "emscripten" or "pyodide" in _geometry_sys.modules


def _geometry_add_path(path):
    text = str(path)
    if text not in _geometry_sys.path:
        _geometry_sys.path.insert(0, text)


def _geometry_find_repo_root():
    candidates = []
    env_root = _geometry_os.environ.get("GEOMETRY_REPO_ROOT")
    if env_root:
        candidates.append(_GeometryPath(env_root).expanduser())
    candidates.append(_GeometryPath.cwd())
    for start in candidates:
        start = start.resolve()
        for current in (start, *start.parents):
            if (current / "course-manifest.json").exists() and (
                current / "metadata" / "runtime_profiles.yml"
            ).exists():
                return current
    raise RuntimeError(
        "Could not find the Geometry repository root. Start JupyterLab inside the "
        "Geometry checkout or set GEOMETRY_REPO_ROOT."
    )


def _geometry_run(command):
    import subprocess as _geometry_subprocess

    printable = " ".join(str(part) for part in command)
    print(f"+ {printable}")
    _geometry_subprocess.check_call([str(part) for part in command])


def _geometry_requirement_names(requirements_path, seen=None):
    seen = set() if seen is None else seen
    requirements_path = requirements_path.resolve()
    if requirements_path in seen or not requirements_path.exists():
        return []
    seen.add(requirements_path)
    names = []
    for raw_line in requirements_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.split("#", 1)[0].strip()
        if not line:
            continue
        if line.startswith(("-r ", "--requirement ")):
            _, nested = line.split(maxsplit=1)
            names.extend(_geometry_requirement_names(requirements_path.parent / nested, seen))
            continue
        if line.startswith("-"):
            continue
        name = line
        for separator in ("==", ">=", "<=", "~=", "!=", ">", "<", ";"):
            name = name.split(separator, 1)[0]
        name = name.split("[", 1)[0].strip()
        if name:
            names.append(name)
    return sorted(set(names))


def _geometry_missing_requirements(requirements_path):
    import importlib.metadata as _geometry_metadata

    missing = []
    for name in _geometry_requirement_names(requirements_path):
        try:
            _geometry_metadata.distribution(name)
        except _geometry_metadata.PackageNotFoundError:
            missing.append(name)
    return missing


def _geometry_configured_roots(repo_root):
    course_dir = GEOMETRY_SETUP.get("course_dir")
    course_root = repo_root / course_dir if course_dir else repo_root
    return repo_root, course_root


if _geometry_is_jupyterlite():
    if not GEOMETRY_SETUP["jupyterlite"]:
        raise RuntimeError(
            "This Geometry notebook uses runtime profile "
            f"{GEOMETRY_SETUP['runtime_profile']!r}, which is not enabled for "
            "JupyterLite in course-manifest.json. Open it in Colab or local JupyterLab."
        )
    GEOMETRY_REPO_ROOT = _GeometryPath.cwd()
    GEOMETRY_COURSE_ROOT = (
        GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["course_dir"]
        if GEOMETRY_SETUP.get("course_dir")
        else GEOMETRY_REPO_ROOT
    )
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    if GEOMETRY_COURSE_ROOT.exists():
        _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        "Geometry setup: JupyterLite/Pyodide detected; shell, git, and pip steps "
        "were skipped."
    )
elif _geometry_is_colab():
    repository = GEOMETRY_SETUP["repository"]
    repo_url = repository["source_url"].rstrip("/") + ".git"
    branch = repository["branch"]
    GEOMETRY_REPO_ROOT = _GeometryPath(
        _geometry_os.environ.get("GEOMETRY_REPO_ROOT", "/content/Geometry")
    )
    sparse_paths = ["requirements", "metadata", "scripts", "course-manifest.json", "index.ipynb"]
    if GEOMETRY_SETUP.get("course_dir"):
        sparse_paths.append(GEOMETRY_SETUP["course_dir"])
    if not (GEOMETRY_REPO_ROOT / ".git").exists():
        if GEOMETRY_REPO_ROOT.exists() and any(GEOMETRY_REPO_ROOT.iterdir()):
            raise RuntimeError(
                f"{GEOMETRY_REPO_ROOT} exists but is not a git checkout. "
                "Set GEOMETRY_REPO_ROOT to an empty path or remove the directory."
            )
        _geometry_run(
            [
                "git",
                "clone",
                "--filter=blob:none",
                "--no-checkout",
                "--branch",
                branch,
                repo_url,
                GEOMETRY_REPO_ROOT,
            ]
        )
        _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "sparse-checkout", "init", "--cone"])
    _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "sparse-checkout", "set", *sparse_paths])
    _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "checkout", branch])
    requirements_path = GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["requirements"]
    _geometry_run([_geometry_sys.executable, "-m", "pip", "install", "-q", "-r", requirements_path])
    GEOMETRY_REPO_ROOT, GEOMETRY_COURSE_ROOT = _geometry_configured_roots(GEOMETRY_REPO_ROOT)
    _geometry_os.chdir(GEOMETRY_COURSE_ROOT if GEOMETRY_COURSE_ROOT.exists() else GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        f"Geometry setup: Colab ready at {_GeometryPath.cwd()} "
        f"with profile {GEOMETRY_RUNTIME_PROFILE!r}."
    )
else:
    GEOMETRY_REPO_ROOT = _geometry_find_repo_root()
    requirements_path = GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["requirements"]
    missing = _geometry_missing_requirements(requirements_path)
    skip_install = _geometry_os.environ.get("GEOMETRY_SKIP_INSTALL") == "1"
    if missing and skip_install:
        print(
            "Geometry setup: GEOMETRY_SKIP_INSTALL=1, so missing profile packages "
            f"were not installed: {', '.join(missing)}"
        )
    elif missing:
        print(
            "Geometry setup: installing missing profile packages from "
            f"{requirements_path.relative_to(GEOMETRY_REPO_ROOT)}: {', '.join(missing)}"
        )
        _geometry_run([_geometry_sys.executable, "-m", "pip", "install", "-r", requirements_path])
    GEOMETRY_REPO_ROOT, GEOMETRY_COURSE_ROOT = _geometry_configured_roots(GEOMETRY_REPO_ROOT)
    _geometry_os.chdir(GEOMETRY_COURSE_ROOT if GEOMETRY_COURSE_ROOT.exists() else GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        f"Geometry setup: local checkout ready at {_GeometryPath.cwd()} "
        f"with profile {GEOMETRY_RUNTIME_PROFILE!r}."
    )


## Translation Guide

| Book concept | Computational representation used here | What to inspect |
| --- | --- | --- |
| Open decomposition $X = U \cup V$ | A two-lobed cover diagram plus an induced group diagram | loops in the overlap have two names before the quotient and one name after it |
| Free product with amalgamation | A normal-closure ledger of relations $i_*(\gamma)j_*(\gamma)^{-1}$ | the quotient identifies the two images of every overlap loop |
| Simply connected intersection | A check where the overlap contributes no nontrivial relation | the union group becomes the free product of the two piece groups |
| One simply connected piece | A quotient of the other piece group by overlap loops | the simply connected piece contributes no generators but can kill loops |
| Wedge sum | Thickened summands around a nondegenerate base point | the base point is made into a contractible overlap so van Kampen applies |
| Finite graph | A NetworkX graph with a spanning tree highlighted | tree edges form a contractible core; each non-tree edge creates one free generator |
| Attaching a 2-cell | A Plotly presentation complex with a boundary word | the attached disk makes that boundary word null-homotopic |
| Attaching an $n$-cell, $n \ge 3$ | A sanity check rather than a new visual | the punctured $n$-ball overlap is simply connected, so $\pi_1$ does not change |
| Surface polygon presentation | A table of generators, relator words, Euler characteristic, and abelianization | orientable surfaces give commutator products; nonorientable surfaces give products of squares |

## Library Routing

NetworkX is the main graph library because this chapter repeatedly converts topology into graph structure: proof dependencies, spanning trees, chords, and finite one-skeleta. Matplotlib is used for durable, labeled PNG diagrams where a static picture is the right artifact: the van Kampen ledger, proof-square sweep, and graph generator picture. Plotly is used for the CW attachment scene because learners benefit from panning and zooming around the wedge loops, polygon boundary, and attaching arrows in a standalone HTML artifact. SymPy is used for exact integer linear algebra in abelianization checks, where relator exponent sums become a matrix over $\mathbb{Z}$. Pandas is used only for compact result tables that make the group-presentation data easy to scan.

## Visual Storyboard

1. **Van Kampen decomposition ledger:** show $U$, $V$, $U \cap V$, the base point, and the induced algebra diagram. Inspection target: the same overlap loop has two names before quotienting. Validation: relation ledger contains the expected normal-closure generator and both special cases.
2. **Proof-square sweep:** draw the homotopy square subdivided into small pieces labeled by whether their images lie in $U$ or $V$. Inspection target: the bottom loop word is pushed row by row to the constant top edge. Validation: every grid cell is assigned to one cover set and the proof-step list has surjectivity, relation containment, and kernel containment.
3. **Finite graph generator system:** choose a spanning tree and color the non-tree edges. Inspection target: free generators are exactly chords outside the tree. Validation: non-tree edge count equals $E - V + 1$.
4. **CW two-cell attachment:** place a wedge of loops next to a polygon boundary word and show the attaching map. Inspection target: a 2-cell kills the boundary word while higher cells do not alter $\pi_1$. Validation: exponent-sum checks for torus and projective-plane relators.
5. **Surface presentation and abelianization table:** compute the abelianized groups from polygon words. Inspection target: orientable genus contributes free rank $2g$, nonorientable genus contributes free rank $n-1$ plus one element of order 2. Validation: Smith normal form gives the listed free ranks and torsion factors.
6. **Applied presentation-complex lab:** treat a finite presentation as a one-vertex CW complex. Inspection target: generators are one-cells, relators are two-cells, and Euler characteristic follows immediately. Validation: known examples match their expected abelianized signatures.


In [ ]:
from pathlib import Path
import json
import math
import os
import sys

import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Ellipse, FancyArrowPatch, Rectangle
import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sympy as sp
from sympy.matrices.normalforms import smith_normal_form
from IPython.display import Markdown, display


def locate_book_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'AGENTS.md').exists() and (candidate / 'source_map.json').exists() and (candidate / 'utils').exists():
            return candidate
    raise RuntimeError('Could not locate Introduction-to-Topological-Manifolds book root')


BOOK_ROOT = locate_book_root()
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import (  # noqa: E402
    assert_artifacts,
    chapter_artifact_root,
    display_artifact,
    save_csv,
    save_json,
    save_matplotlib,
    save_plotly_html,
)
from utils.source import unit_by_artifact_key  # noqa: E402
from utils.topology import cycle_rank_for_graph, euler_characteristic, word_reduce  # noqa: E402
from utils.validation import image_stats, relative  # noqa: E402

UNIT_KEY = 'chapter-10-the-seifert-van-kampen-theorem'
unit = unit_by_artifact_key(UNIT_KEY)
ARTIFACT_ROOT = chapter_artifact_root(UNIT_KEY, BOOK_ROOT)
FIGURES = ARTIFACT_ROOT / 'figures'
HTML = ARTIFACT_ROOT / 'html'
CHECKS = ARTIFACT_ROOT / 'checks'
TABLES = ARTIFACT_ROOT / 'tables'

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'font.size': 10,
})

source_span = {
    'printed_pages': unit['printed'],
    'pdf_pages': unit['pdf'],
    'pdftotext_pages_inspected_for_printed_span': 'physical pages 267-291 in this PDF align with printed pp. 251-275; the assigned source map records PDF pp. 269-294',
}
print(f"Book root: {BOOK_ROOT.name}")
print(f"Artifact root: {relative(ARTIFACT_ROOT)}")


In [ ]:
storyboard = {
    'chapter_goal': 'Compute fundamental groups from gluing data: van Kampen decompositions, finite graphs, finite CW complexes, and compact surface polygon words.',
    'source_span_read': 'Chapter 10, printed pp. 251-276, PDF pp. 269-294; inspected with pdftotext for structure and terminology only.',
    'library_routing': [
        {'concept': 'van Kampen decomposition', 'representation': 'cover plus induced group diagram', 'library': 'matplotlib, networkx', 'why': 'static labeled arrows make the quotient ledger inspectable'},
        {'concept': 'proof scaffold', 'representation': 'subdivided homotopy square', 'library': 'matplotlib', 'why': 'the square grid is the core geometric move in the kernel argument'},
        {'concept': 'finite graph group', 'representation': 'spanning tree and chords', 'library': 'networkx, matplotlib', 'why': 'spanning trees and non-tree edges are graph-native structures'},
        {'concept': 'CW two-cell attachment', 'representation': 'interactive presentation complex', 'library': 'plotly', 'why': 'zoomable HTML keeps the wedge loops, polygon word, and attaching map in one view'},
        {'concept': 'surface group presentations', 'representation': 'exact abelianization table', 'library': 'sympy, pandas', 'why': 'relator exponent matrices and Smith normal form are integer algebra'},
    ],
    'visual_sequence': [
        {'order': 1, 'artifact': 'figures/van-kampen-decomposition-ledger.png', 'inspection_target': 'overlap loops have two names before quotienting', 'validation': 'normal-closure relation recorded'},
        {'order': 2, 'artifact': 'figures/van-kampen-proof-grid-sweep.png', 'inspection_target': 'homotopy square moves a loop word row by row', 'validation': 'all cells assigned to U or V'},
        {'order': 3, 'artifact': 'figures/graph-spanning-tree-generators.png', 'inspection_target': 'non-tree edges are free generators', 'validation': 'cycle rank equals chord count'},
        {'order': 4, 'artifact': 'html/cw-two-cell-attachment.html', 'inspection_target': 'a disk kills its boundary word', 'validation': 'relator exponent sums checked'},
        {'order': 5, 'artifact': 'tables/surface-group-presentations.csv', 'inspection_target': 'surface polygon words become group relators', 'validation': 'abelianization signatures checked'},
        {'order': 6, 'artifact': 'tables/applied-lab-presentation-complexes.csv', 'inspection_target': 'finite presentations as one-vertex CW complexes', 'validation': 'Euler characteristic and abelianization examples checked'},
    ],
    'computational_checks': [
        'artifact existence and nonzero size',
        'PNG image nonblank statistics',
        'graph cycle rank E - V + 1 equals number of non-tree edges',
        'relator exponent sums and Smith normal form for surface abelianizations',
        'source map span matches assigned chapter metadata',
    ],
}
storyboard_path = save_json(storyboard, CHECKS / 'visual-storyboard.json')
display_artifact(storyboard_path)


## Concept Section 1: Van Kampen as a Gluing Ledger

The theorem begins with a cover $X = U \cup V$ where $U$, $V$, and $U \cap V$ are path-connected and the base point lies in the overlap. A loop in $X$ can be subdivided so that each short segment lies in one side of the cover. After adding connector paths back to the base point, each short segment becomes a loop in $U$ or a loop in $V$. This explains why the free product $\pi_1(U,p) * \pi_1(V,p)$ maps onto $\pi_1(X,p)$: every loop in the union can be described by alternating letters from the two piece groups.

Surjectivity is not the whole story. If a loop $\gamma$ lives in the overlap, it can be read as a loop in $U$ or as a loop in $V$. These are different letters in the free product, but they become the same loop after both are included into $X$. The theorem says that this is the only ambiguity: quotient the free product by the normal closure of all words $i_*(\gamma)j_*(\gamma)^{-1}$ coming from overlap loops, and the result is $\pi_1(X,p)$.

Two special cases are especially useful. If $U \cap V$ is simply connected, then it contributes no nontrivial overlap loops, so the group of the union is the free product of the two piece groups. If $U$ is simply connected, then $U$ contributes no generators, but the overlap can still impose relations on $\pi_1(V,p)$. The notebook will reuse exactly those two reductions for wedge sums, graphs, and cell attachments.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.4), constrained_layout=True)
ax = axes[0]
ax.set_title('Open cover of X')
ax.set_aspect('equal')
ax.axis('off')
left = Ellipse((-0.45, 0), 1.8, 1.25, angle=12, facecolor='#8ecae6', edgecolor='#126782', alpha=0.68, lw=2)
right = Ellipse((0.45, 0), 1.8, 1.25, angle=-12, facecolor='#ffb703', edgecolor='#b45309', alpha=0.60, lw=2)
ax.add_patch(left)
ax.add_patch(right)
ax.add_patch(Circle((0, 0), 0.08, facecolor='#111827', edgecolor='white', lw=1.5, zorder=5))
ax.text(-0.8, 0.42, 'U', fontsize=16, weight='bold', color='#0f4c5c')
ax.text(0.78, 0.42, 'V', fontsize=16, weight='bold', color='#7c2d12')
ax.text(-0.18, -0.16, 'p in U cap V', fontsize=10, color='#111827')
ax.text(-0.2, 0.58, 'overlap\nloops gamma', ha='center', va='center', fontsize=10)
theta = np.linspace(0, 2 * np.pi, 200)
ax.plot(0.25 * np.cos(theta), 0.18 * np.sin(theta) + 0.18, color='#6d28d9', lw=2)
ax.set_xlim(-1.6, 1.6)
ax.set_ylim(-1.0, 1.0)

ax = axes[1]
ax.set_title('Induced group ledger')
ax.axis('off')
G = nx.DiGraph()
labels = {
    'I': r'$\pi_1(U\cap V,p)$',
    'U': r'$\pi_1(U,p)$',
    'V': r'$\pi_1(V,p)$',
    'F': r'$\pi_1(U,p)*\pi_1(V,p)$',
    'Q': r'$(* )/<<C>>$',
    'X': r'$\pi_1(X,p)$',
}
G.add_edges_from([
    ('I', 'U'), ('I', 'V'), ('U', 'F'), ('V', 'F'), ('F', 'Q'), ('Q', 'X')
])
pos = {
    'I': (0.0, 0.0), 'U': (1.15, 0.9), 'V': (1.15, -0.9),
    'F': (2.75, 0.0), 'Q': (4.35, 0.0), 'X': (5.95, 0.0),
}
edge_labels = {
    ('I', 'U'): r'$i_*$', ('I', 'V'): r'$j_*$', ('U', 'F'): 'include',
    ('V', 'F'): 'include', ('F', 'Q'): 'quotient', ('Q', 'X'): r'$\cong$',
}
nx.draw_networkx_nodes(G, pos, node_size=2300, node_color=['#dbeafe', '#dbeafe', '#fef3c7', '#ede9fe', '#dcfce7', '#fee2e2'], edgecolors='#111827', ax=ax)
nx.draw_networkx_labels(G, pos, labels=labels, font_size=9, ax=ax)
nx.draw_networkx_edges(G, pos, arrows=True, arrowstyle='-|>', arrowsize=16, width=1.6, edge_color='#374151', ax=ax)
for edge, text in edge_labels.items():
    x0, y0 = pos[edge[0]]
    x1, y1 = pos[edge[1]]
    x = 0.52 * x0 + 0.48 * x1
    y = 0.52 * y0 + 0.48 * y1
    if edge in {('F', 'Q'), ('Q', 'X')}:
        y += 0.22
    ax.text(x, y, text, ha='center', va='center', fontsize=8, bbox={'boxstyle': 'round,pad=0.15', 'facecolor': 'white', 'edgecolor': 'none', 'alpha': 0.85})
ax.text(4.35, -0.48, r'$C=\{i_*(\gamma)j_*(\gamma)^{-1}\}$', ha='center', fontsize=9)
ax.text(3.05, -1.3, 'Normal closure records every overlap identification.', ha='center', fontsize=9)
ax.set_xlim(-0.55, 6.45)
ax.set_ylim(-1.6, 1.4)

van_kampen_relations = {
    'cover_assumptions': ['U open', 'V open', 'U union V is X', 'U path-connected', 'V path-connected', 'U cap V path-connected', 'p in U cap V'],
    'free_product_before_quotient': 'pi1(U,p) * pi1(V,p)',
    'overlap_relation_template': 'i_*(gamma) * j_*(gamma)^-1',
    'quotient_after_relations': 'pi1(X,p)',
    'special_cases': {
        'simply_connected_intersection': 'pi1(X,p) is pi1(U,p) * pi1(V,p)',
        'one_simply_connected_piece': 'pi1(X,p) is the other piece group modulo the normal closure of overlap loops',
    },
}
van_kampen_png = save_matplotlib(fig, FIGURES / 'van-kampen-decomposition-ledger.png')
van_kampen_json = save_json(van_kampen_relations, CHECKS / 'van-kampen-relations.json')
plt.close(fig)
display_artifact(van_kampen_png)
display_artifact(van_kampen_json)


## Proof and Invariant Scaffold: Why the Ledger Has No Missing Terms

The proof has three computationally meaningful stages. First, every loop in the union is subdivided into small pieces that lie entirely inside one cover set. The connectors from the base point turn those pieces into based loops, so the free product map is surjective. Second, every overlap loop gives an element of the kernel, because including it through $U$ or through $V$ gives the same loop in $X$. Third, if a free-product word maps to the identity in $X$, choose a null-homotopy and subdivide the homotopy square finely enough that each small square maps into $U$ or into $V$. Then sweep the bottom word upward one row at a time. Whenever a small square crosses from one cover label to the other, the difference is exactly one of the overlap relations already placed in the normal closure.

The figure below is not a proof by itself. It is a bookkeeping scaffold. Read the bottom edge as the loop word we start with, the top edge as the constant loop, and the colored grid as the Lebesgue-number subdivision of the null-homotopy. The vertical edges are the connector paths used to rebase the small path segments. The invariant being tracked is the coset modulo the normal closure of the overlap relations. Each row replacement preserves that coset, so a word in the kernel becomes equivalent to the identity.


In [ ]:
rows, cols = 4, 7
cell_labels = np.array([
    ['U', 'U', 'V', 'V', 'U', 'V', 'V'],
    ['U', 'V', 'V', 'U', 'U', 'U', 'V'],
    ['V', 'V', 'U', 'U', 'V', 'U', 'U'],
    ['V', 'U', 'U', 'V', 'V', 'U', 'U'],
])
colors = {'U': '#bfdbfe', 'V': '#fde68a'}
fig, ax = plt.subplots(figsize=(9, 5), constrained_layout=True)
ax.set_title('Null-homotopy square subdivided by the open cover')
ax.set_aspect('equal')
for j in range(rows):
    for i in range(cols):
        label = cell_labels[j, i]
        rect = Rectangle((i, j), 1, 1, facecolor=colors[label], edgecolor='#374151', lw=1.0)
        ax.add_patch(rect)
        ax.text(i + 0.5, j + 0.5, label, ha='center', va='center', fontsize=12, weight='bold')
for i in range(cols + 1):
    ax.plot([i, i], [0, rows], color='#6b7280', lw=0.6)
for j in range(rows + 1):
    ax.plot([0, cols], [j, j], color='#6b7280', lw=0.6)
for i in range(cols):
    ax.add_patch(FancyArrowPatch((i + 0.15, -0.25), (i + 0.85, -0.25), arrowstyle='-|>', mutation_scale=12, color='#1f2937'))
    ax.text(i + 0.5, -0.5, f'a{i+1}', ha='center', fontsize=9)
for j in range(rows):
    ax.add_patch(FancyArrowPatch((cols + 0.35, j + 0.15), (cols + 0.35, j + 0.85), arrowstyle='-|>', mutation_scale=14, color='#7c3aed'))
ax.plot([0, cols], [rows + 0.25, rows + 0.25], color='#111827', lw=2)
ax.text(cols / 2, rows + 0.45, 'top edge is the constant loop', ha='center', fontsize=10)
ax.text(-0.15, -0.25, 'bottom word', ha='right', va='center', fontsize=10)
ax.text(cols + 0.55, rows / 2, 'row sweep\nmod overlap\nrelations', ha='left', va='center', fontsize=10, color='#4c1d95')
ax.set_xlim(-0.9, cols + 1.7)
ax.set_ylim(-0.8, rows + 0.8)
ax.axis('off')

proof_scaffold = {
    'grid_shape': [int(rows), int(cols)],
    'cell_counts': {label: int((cell_labels == label).sum()) for label in ['U', 'V']},
    'proof_steps': ['surjectivity by loop subdivision', 'overlap relations lie in the kernel', 'kernel word swept to identity through a square grid'],
    'invariant_tracked': 'coset modulo the normal closure of i_*(gamma)j_*(gamma)^-1',
    'all_cells_labeled': bool(np.isin(cell_labels, ['U', 'V']).all()),
}
proof_grid_png = save_matplotlib(fig, FIGURES / 'van-kampen-proof-grid-sweep.png')
proof_grid_json = save_json(proof_scaffold, CHECKS / 'van-kampen-proof-grid-check.json')
plt.close(fig)
display_artifact(proof_grid_png)


## Concept Section 2: Wedge Sums and Finite Graphs

A wedge sum is the first application where the theorem has to be used with care. The summands meet at a single base point, but the summands themselves are usually not open in the wedge. The nondegenerate base point condition fixes that problem: thicken each summand by a small neighborhood inside the other summand, and the overlap becomes contractible. The simply connected intersection case of van Kampen then gives a free product. For a bouquet of $n$ circles, this recovers the free group on $n$ visible loop generators.

Finite graphs turn that observation into a repeatable algorithm. A tree is contractible, so it contributes no fundamental group. Every finite connected graph has a spanning tree, and the edges outside the tree are exactly the extra loops that cannot be collapsed away. If the graph has $V$ vertices, $E$ edges, and one connected component, the number of non-tree edges is $E - V + 1$. That is both the cycle rank of the graph and the rank of its free fundamental group.

The graph below is deliberately drawn with a chosen spanning tree rather than with a generic layout. Gray edges form the contractible core. Colored chord edges are the free generators. Each generator is read as: start at the base vertex, move through the tree to one endpoint of the chord, traverse the chord, and return through the tree. Different choices of tree paths give homotopic loops because the tree is contractible.


In [ ]:
vertices = list(range(6))
tree_edges = [(0, 1), (1, 2), (1, 3), (3, 4), (3, 5)]
chord_edges = [(2, 4), (0, 5), (2, 5)]
all_edges = tree_edges + chord_edges
Gamma = nx.Graph()
Gamma.add_nodes_from(vertices)
Gamma.add_edges_from(all_edges)
T = nx.Graph()
T.add_nodes_from(vertices)
T.add_edges_from(tree_edges)
assert nx.is_tree(T)
assert nx.is_connected(Gamma)

rank = cycle_rank_for_graph(Gamma.number_of_nodes(), Gamma.number_of_edges(), nx.number_connected_components(Gamma))
pos = {0: (0, 0), 1: (1, 0.5), 2: (2, 0.8), 3: (1, -0.7), 4: (2.15, -0.35), 5: (0.05, -1.25)}
fig, ax = plt.subplots(figsize=(8, 5.5), constrained_layout=True)
ax.set_title('Finite graph = spanning tree core + free generators')
ax.axis('off')
nx.draw_networkx_edges(Gamma, pos, edgelist=tree_edges, width=4, edge_color='#9ca3af', ax=ax)
chord_colors = ['#dc2626', '#2563eb', '#16a34a']
for idx, edge in enumerate(chord_edges, start=1):
    nx.draw_networkx_edges(Gamma, pos, edgelist=[edge], width=3, edge_color=chord_colors[idx - 1], style='dashed', ax=ax)
    x = (pos[edge[0]][0] + pos[edge[1]][0]) / 2
    y = (pos[edge[0]][1] + pos[edge[1]][1]) / 2
    ax.text(x, y + 0.12, f'f{idx}', color=chord_colors[idx - 1], fontsize=12, weight='bold', ha='center')
nx.draw_networkx_nodes(Gamma, pos, node_size=650, node_color='#f9fafb', edgecolors='#111827', linewidths=1.5, ax=ax)
nx.draw_networkx_labels(Gamma, pos, font_size=10, ax=ax)
ax.scatter([pos[0][0]], [pos[0][1]], s=180, color='#111827', zorder=5)
ax.text(pos[0][0] - 0.1, pos[0][1] + 0.22, 'base vertex', ha='right', fontsize=9)
ax.text(1.1, -1.55, f'V={Gamma.number_of_nodes()}, E={Gamma.number_of_edges()}, cycle rank E - V + 1 = {rank}', ha='center', fontsize=10)

graph_summary = {
    'vertices': Gamma.number_of_nodes(),
    'edges': Gamma.number_of_edges(),
    'component_count': nx.number_connected_components(Gamma),
    'spanning_tree_edges': [list(edge) for edge in tree_edges],
    'non_tree_edges': [list(edge) for edge in chord_edges],
    'cycle_rank': int(rank),
    'free_group_generators': [f'f{i}' for i in range(1, len(chord_edges) + 1)],
    'rank_equals_non_tree_edges': bool(rank == len(chord_edges)),
}
graph_png = save_matplotlib(fig, FIGURES / 'graph-spanning-tree-generators.png')
graph_json = save_json(graph_summary, CHECKS / 'graph-spanning-tree-generators.json')
plt.close(fig)
display_artifact(graph_png)
display(Markdown(f"Graph check: free rank `{rank}` equals the `{len(chord_edges)}` highlighted non-tree edges."))


## Concept Section 3: CW Complexes as Presentation Machines

A finite CW complex lets us compute $\pi_1$ in layers. The zero-cells and one-cells form a finite graph, so the one-skeleton has a free fundamental group once we choose a spanning tree. A two-cell is different: its attaching map is a loop in the one-skeleton. After the disk is attached, that loop bounds the new disk, so it becomes null-homotopic. Algebraically, a two-cell adds one relator. Attaching several two-cells adds several relators, one for each boundary word.

The visual below uses the torus relation as a model. Start with a wedge of two circles, one generator $a$ and one generator $b$. The boundary of a square reads $aba^{-1}b^{-1}$. Attaching the square along that loop kills the commutator. The resulting presentation has two generators and one commutator relator, so its abelianization is free abelian of rank two. That agrees with the geometric idea that a torus has two independent one-dimensional directions.

For cells in dimension at least three, the fundamental group does not change. The reason is another special case of van Kampen: after deleting an interior point from the new cell, the relevant overlap is simply connected when the cell dimension is at least three. There is no new loop relation to add. Thus, for $\pi_1$, the two-skeleton contains all the information in a finite CW complex.


In [ ]:
def is_inverse_token(token):
    return token.endswith('^-1')


def token_base(token):
    return token[:-3] if is_inverse_token(token) else token


def token_sign(token):
    return -1 if is_inverse_token(token) else 1


def inverse_token(token):
    return token_base(token) if is_inverse_token(token) else f'{token}^-1'


def exponent_sums(word, generators):
    counts = {generator: 0 for generator in generators}
    for token in word:
        counts[token_base(token)] += token_sign(token)
    return counts


def word_to_text(word):
    return ' '.join(word) if word else '1'


torus_word = ['a', 'b', 'a^-1', 'b^-1']
projective_word = ['x', 'x']

fig_cw = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=('one-skeleton: wedge of loops', 'attaching disk boundary word'),
    specs=[[{'type': 'xy'}, {'type': 'xy'}]],
    horizontal_spacing=0.16,
)
t = np.linspace(0, 2 * np.pi, 240)
fig_cw.add_trace(go.Scatter(x=-0.8 + 0.8 * np.cos(t), y=0.8 * np.sin(t), mode='lines', name='a loop', line={'color': '#2563eb', 'width': 4}), row=1, col=1)
fig_cw.add_trace(go.Scatter(x=0.8 - 0.8 * np.cos(t), y=0.8 * np.sin(t), mode='lines', name='b loop', line={'color': '#dc2626', 'width': 4}), row=1, col=1)
fig_cw.add_trace(go.Scatter(x=[0], y=[0], mode='markers+text', text=['base'], textposition='bottom center', marker={'size': 10, 'color': '#111827'}, name='base point'), row=1, col=1)
fig_cw.add_annotation(x=-1.55, y=0.1, text='a', showarrow=False, font={'color': '#2563eb', 'size': 16}, row=1, col=1)
fig_cw.add_annotation(x=1.55, y=0.1, text='b', showarrow=False, font={'color': '#dc2626', 'size': 16}, row=1, col=1)
sides = [
    ([0, 1], [0, 0], 'a', '#2563eb'),
    ([1, 1], [0, 1], 'b', '#dc2626'),
    ([1, 0], [1, 1], 'a^-1', '#2563eb'),
    ([0, 0], [1, 0], 'b^-1', '#dc2626'),
]
for xs, ys, label, color in sides:
    fig_cw.add_trace(go.Scatter(x=xs, y=ys, mode='lines', line={'color': color, 'width': 5}, name=label), row=1, col=2)
fig_cw.add_trace(go.Scatter(x=[0.5], y=[0.5], mode='markers+text', text=['2-cell'], textposition='middle center', marker={'size': 90, 'color': 'rgba(22,163,74,0.25)'}, name='attached disk'), row=1, col=2)
for x, y, label, color in [(0.5, -0.12, 'a', '#2563eb'), (1.12, 0.5, 'b', '#dc2626'), (0.5, 1.12, 'a^-1', '#2563eb'), (-0.15, 0.5, 'b^-1', '#dc2626')]:
    fig_cw.add_annotation(x=x, y=y, text=label, showarrow=False, font={'color': color, 'size': 14}, row=1, col=2)
fig_cw.update_xaxes(visible=False, scaleanchor='y', scaleratio=1)
fig_cw.update_yaxes(visible=False)
fig_cw.update_layout(
    title='A 2-cell adds the relator read from its boundary word',
    showlegend=True,
    width=920,
    height=460,
    margin={'l': 20, 'r': 20, 't': 70, 'b': 20},
)

cw_checks = {
    'torus_boundary_word': torus_word,
    'torus_exponent_sums': exponent_sums(torus_word, ['a', 'b']),
    'projective_plane_boundary_word': projective_word,
    'projective_plane_exponent_sums': exponent_sums(projective_word, ['x']),
    'two_cell_effect': 'add the boundary word as a relator',
    'n_cell_for_n_at_least_3_effect_on_pi1': 'no change',
    'torus_commutator_reduces_freely_to': word_reduce(torus_word),
}
cw_html = save_plotly_html(fig_cw, HTML / 'cw-two-cell-attachment.html')
cw_json = save_json(cw_checks, CHECKS / 'cw-attachment-relation-check.json')
display_artifact(cw_html, width=900, height=480)
display(Markdown(f"Torus boundary word: `{word_to_text(torus_word)}`. Its abelian exponent sums are `{cw_checks['torus_exponent_sums']}`."))


## Concept Section 4: Surface Group Presentations

A polygonal presentation of a compact surface is already close to a group presentation. When all polygon vertices are identified to one point, the one-skeleton is a wedge of circles, one for each edge label. The single polygon face is a two-cell. Its boundary word becomes the relator. Thus the surface group is read directly from the polygon word: generators are edge labels, and the relator is the cyclic word around the boundary.

For an orientable surface of genus $g$, the standard word is a product of commutators. In the abelianization, every commutator has zero exponent sum, so the abelianized group is $\mathbb{Z}^{2g}$. For a connected sum of $n$ projective planes, the standard word is a product of squares. In the abelianization, the single relation says $2x_1 + \cdots + 2x_n = 0$. Smith normal form converts that relation to one torsion factor of order two and $n-1$ free directions.

This is the algebraic reason fundamental groups finish the compact-surface classification. The sphere has trivial fundamental group. Orientable surfaces have torsion-free abelianizations of even rank. Nonorientable surfaces have a visible element of order two in the abelianization. Within each family, the free rank determines the genus. The table below computes these facts from relator words rather than treating them as labels.


In [ ]:
def relation_matrix(relator_words, generators):
    if not relator_words:
        return sp.zeros(0, len(generators))
    rows = []
    for word in relator_words:
        sums = exponent_sums(word, generators)
        rows.append([sums[generator] for generator in generators])
    return sp.Matrix(rows)


def abelianization_summary(generators, relator_words):
    matrix = relation_matrix(relator_words, generators)
    generator_count = len(generators)
    if matrix.rows == 0 or matrix.cols == 0:
        rank = 0
        torsion = []
        smith_diagonal = []
    else:
        smith = smith_normal_form(matrix, domain=sp.ZZ)
        diagonal = [abs(int(smith[i, i])) for i in range(min(smith.rows, smith.cols))]
        smith_diagonal = [d for d in diagonal if d != 0]
        rank = len(smith_diagonal)
        torsion = [d for d in smith_diagonal if d > 1]
    free_rank = generator_count - rank
    return {
        'generator_count': generator_count,
        'relation_matrix': [[int(matrix[i, j]) for j in range(matrix.cols)] for i in range(matrix.rows)],
        'smith_diagonal': smith_diagonal,
        'free_rank': int(free_rank),
        'torsion_factors': torsion,
    }


def commutator(a, b):
    return [a, b, f'{a}^-1', f'{b}^-1']


def orientable_surface_case(genus):
    generators = []
    word = []
    for i in range(1, genus + 1):
        a, b = f'a{i}', f'b{i}'
        generators.extend([a, b])
        word.extend(commutator(a, b))
    return generators, [word]


def nonorientable_surface_case(genus):
    generators = [f'x{i}' for i in range(1, genus + 1)]
    word = []
    for generator in generators:
        word.extend([generator, generator])
    return generators, [word]

surface_specs = [
    {'surface': 'sphere', 'family': 'orientable', 'genus_parameter': 0, 'generators': [], 'relators': [], 'expected_free_rank': 0, 'expected_torsion': []},
    {'surface': 'torus', 'family': 'orientable', 'genus_parameter': 1, 'expected_free_rank': 2, 'expected_torsion': []},
    {'surface': 'orientable genus 2', 'family': 'orientable', 'genus_parameter': 2, 'expected_free_rank': 4, 'expected_torsion': []},
    {'surface': 'projective plane', 'family': 'nonorientable', 'genus_parameter': 1, 'expected_free_rank': 0, 'expected_torsion': [2]},
    {'surface': 'Klein bottle', 'family': 'nonorientable', 'genus_parameter': 2, 'expected_free_rank': 1, 'expected_torsion': [2]},
    {'surface': 'nonorientable genus 3', 'family': 'nonorientable', 'genus_parameter': 3, 'expected_free_rank': 2, 'expected_torsion': [2]},
]

surface_rows = []
surface_checks = {}
for spec in surface_specs:
    if 'generators' in spec:
        generators, relators = spec['generators'], spec['relators']
    elif spec['family'] == 'orientable':
        generators, relators = orientable_surface_case(spec['genus_parameter'])
    else:
        generators, relators = nonorientable_surface_case(spec['genus_parameter'])
    summary = abelianization_summary(generators, relators)
    chi = 2 - 2 * spec['genus_parameter'] if spec['family'] == 'orientable' else 2 - spec['genus_parameter']
    relator_text = 'none' if spec['surface'] == 'sphere' else '; '.join(word_to_text(word) for word in relators)
    row = {
        'surface': spec['surface'],
        'family': spec['family'],
        'genus_parameter': spec['genus_parameter'],
        'generators': ', '.join(generators) if generators else 'none',
        'relator_word': relator_text,
        'euler_characteristic': chi,
        'abelian_free_rank': summary['free_rank'],
        'abelian_torsion_factors': ', '.join(map(str, summary['torsion_factors'])) if summary['torsion_factors'] else 'none',
    }
    surface_rows.append(row)
    surface_checks[spec['surface']] = {
        **summary,
        'expected_free_rank': spec['expected_free_rank'],
        'expected_torsion_factors': spec['expected_torsion'],
        'matches_expected': bool(summary['free_rank'] == spec['expected_free_rank'] and summary['torsion_factors'] == spec['expected_torsion']),
    }

surface_table_path = save_csv(surface_rows, TABLES / 'surface-group-presentations.csv')
surface_checks_path = save_json(surface_checks, CHECKS / 'surface-abelianization-checks.json')
surface_df = pd.DataFrame(surface_rows)
display(surface_df)
display_artifact(surface_table_path)


## Applied Lab: Build a Presentation Complex and Read Its First Invariants

A finite presentation can be visualized as a one-vertex CW complex: one loop edge for each generator and one attached disk for each relator. This construction is a practical bridge between algebra and topology. If a space is built as a finite two-dimensional CW complex, van Kampen gives a presentation for its fundamental group. Conversely, a finite presentation can be realized by a finite two-complex. The topology gives a way to draw and test the algebra; the algebra gives a compact way to compare spaces.

The lab cell below packages this construction. Each row is a small presentation complex. The Euler characteristic is computed from the cells: one vertex, one edge per generator, and one face per relator. Abelianization is computed by sending each relator word to its exponent-sum row and taking Smith normal form. This does not solve the full group isomorphism problem, but it is exactly the invariant used in this chapter to distinguish compact surface families.

Try modifying the `lab_presentations` dictionary after running the notebook. Add a new generator, add a relator, or replace a commutator with a square. The table will immediately show which first abelian invariant changed and which did not. That feedback mirrors the chapter's main lesson: van Kampen turns geometric gluing data into algebraic constraints that can be inspected and checked.


In [ ]:
lab_presentations = {
    'bouquet of three circles': {
        'generators': ['a', 'b', 'c'],
        'relators': [],
        'expected': {'free_rank': 3, 'torsion_factors': []},
    },
    'torus presentation complex': {
        'generators': ['a', 'b'],
        'relators': [commutator('a', 'b')],
        'expected': {'free_rank': 2, 'torsion_factors': []},
    },
    'projective plane presentation complex': {
        'generators': ['x'],
        'relators': [['x', 'x']],
        'expected': {'free_rank': 0, 'torsion_factors': [2]},
    },
    'orientable genus 2 presentation complex': {
        'generators': orientable_surface_case(2)[0],
        'relators': orientable_surface_case(2)[1],
        'expected': {'free_rank': 4, 'torsion_factors': []},
    },
    'nonorientable genus 3 presentation complex': {
        'generators': nonorientable_surface_case(3)[0],
        'relators': nonorientable_surface_case(3)[1],
        'expected': {'free_rank': 2, 'torsion_factors': [2]},
    },
}

lab_rows = []
lab_checks = {}
for name, data in lab_presentations.items():
    generators = data['generators']
    relators = data['relators']
    summary = abelianization_summary(generators, relators)
    vertices = 1
    edges = len(generators)
    faces = len(relators)
    chi = euler_characteristic(vertices, edges, faces)
    expected = data['expected']
    matches = bool(summary['free_rank'] == expected['free_rank'] and summary['torsion_factors'] == expected['torsion_factors'])
    lab_rows.append({
        'presentation_complex': name,
        'cells_V_E_F': f'{vertices},{edges},{faces}',
        'euler_characteristic': chi,
        'relators': '; '.join(word_to_text(word) for word in relators) if relators else 'none',
        'abelian_free_rank': summary['free_rank'],
        'torsion_factors': ', '.join(map(str, summary['torsion_factors'])) if summary['torsion_factors'] else 'none',
        'matches_expected': matches,
    })
    lab_checks[name] = {
        'vertices': vertices,
        'edges': edges,
        'faces': faces,
        'euler_characteristic': chi,
        'abelianization': summary,
        'expected': expected,
        'matches_expected': matches,
    }

lab_table_path = save_csv(lab_rows, TABLES / 'applied-lab-presentation-complexes.csv')
lab_checks_path = save_json(lab_checks, CHECKS / 'applied-lab-presentation-complexes.json')
lab_df = pd.DataFrame(lab_rows)
display(lab_df)
display_artifact(lab_table_path)


## Final Sanity Checks

The final cell checks both the mathematics and the artifact contract. It verifies that the source-map metadata is still the assigned chapter span, every generated artifact exists and is nonempty, both PNGs have nontrivial image variation, the graph rank formula matches the highlighted generators, the surface abelianization rows match their expected free ranks and torsion factors, and the applied lab examples pass their invariant checks. It then saves a compact JSON report under the chapter check artifacts.


In [ ]:
artifact_paths = [
    storyboard_path,
    van_kampen_png,
    van_kampen_json,
    proof_grid_png,
    proof_grid_json,
    graph_png,
    graph_json,
    cw_html,
    cw_json,
    surface_table_path,
    surface_checks_path,
    lab_table_path,
    lab_checks_path,
]
assert_artifacts(artifact_paths, min_bytes=64)

png_stats = [image_stats(path) for path in [van_kampen_png, proof_grid_png, graph_png]]
assert all(item['width'] >= 600 and item['height'] >= 350 for item in png_stats)
assert all(item['max_channel_stddev'] > 5.0 for item in png_stats)
assert graph_summary['rank_equals_non_tree_edges']
assert graph_summary['cycle_rank'] == len(graph_summary['non_tree_edges'])
assert van_kampen_relations['overlap_relation_template'] == 'i_*(gamma) * j_*(gamma)^-1'
assert proof_scaffold['all_cells_labeled']
assert len(storyboard['visual_sequence']) >= 5
assert source_span['printed_pages'] == '251-276'
assert source_span['pdf_pages'] == '269-294'
assert all(item['matches_expected'] for item in surface_checks.values())
assert all(item['matches_expected'] for item in lab_checks.values())
assert cw_checks['torus_exponent_sums'] == {'a': 0, 'b': 0}
assert cw_checks['projective_plane_exponent_sums'] == {'x': 2}

final_sanity = {
    'source_span': source_span,
    'artifact_count': len(artifact_paths),
    'artifacts': [relative(path) for path in artifact_paths],
    'png_stats': png_stats,
    'graph_cycle_rank': graph_summary['cycle_rank'],
    'surface_checks_passed': all(item['matches_expected'] for item in surface_checks.values()),
    'lab_checks_passed': all(item['matches_expected'] for item in lab_checks.values()),
    'cw_checks': cw_checks,
}
final_sanity_path = save_json(final_sanity, CHECKS / 'final-sanity.json')
assert_artifacts([final_sanity_path], min_bytes=64)
display_artifact(final_sanity_path)
display(Markdown(f"Final sanity passed for `{len(artifact_paths) + 1}` artifacts."))


## Takeaways

The Seifert-Van Kampen theorem is a controlled way to compute a fundamental group from a cover. The free product records loops that live in either open set; the overlap relations identify the two ways of naming a loop that already lies in the intersection. The two special cases are the workhorses: a simply connected overlap gives a free product, while a simply connected piece turns the other piece group into a quotient.

Graphs and CW complexes are the cleanest computational applications. A finite connected graph has a contractible spanning tree, and each non-tree edge gives one free generator. A finite CW complex gets its $\pi_1$ from the one-skeleton and the two-cells: one-skeleton generators first, then one relator for each attached two-cell. Higher-dimensional cells do not alter $\pi_1$.

Compact surface groups are polygon words made algebraic. Orientable genus $g$ surfaces have presentations with $2g$ generators and one product-of-commutators relator, whose abelianization is $\mathbb{Z}^{2g}$. Nonorientable genus $n$ surfaces have a product-of-squares relator, whose abelianization is $\mathbb{Z}^{n-1} \oplus \mathbb{Z}/2$. Those abelianized signatures separate the surface families and recover the genus within each family.
